In [ ]:
import inspect
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gymnasium import spaces
import torch
from IPython.display import display

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'environment').exists() and (candidate / 'experts').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root.')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import IQN allocator instead of PPO
from allocator.allocator_agent import IQNAllocator
from allocator.allocator_env import AllocatorEnv
from allocator.train_allocator import (
    build_base_env,
    compute_max_drawdown,
    compute_sharpe,
    compute_sortino,
    load_config,
    load_expert_agent,
)
from experts.train_experts import EXPERT_NAME_MAP, detect_asset_type
from regime.config import CONFIG as REGIME_CONFIG
from regime.inference import MultiHorizonRegimeClassifier

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
ASSET = 'EURUSD'  # Change this to the desired asset (e.g., 'EURUSD', 'BTCUSDT', etc.)
DEVICE = 'cpu'

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'allocator.yaml'
EXPERTS_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'experts.yaml'
TEST_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / ASSET / 'train.csv'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / ASSET / 'diversity'
ALLOCATOR_PATH = CHECKPOINT_DIR  / 'allocator.pt'         
REGIME_MODEL_PATH = CHECKPOINT_DIR.parent / 'regime' / f'{ASSET}_tcn_multi.pth'

alloc_cfg = load_config(CONFIG_PATH)
experts_cfg = load_config(EXPERTS_CONFIG_PATH)
features = list(experts_cfg['ppo']['features'])
asset_type = detect_asset_type(ASSET)
lookback = int(alloc_cfg['allocator'].get('lookback', 20))

test_df = pd.read_csv(TEST_DATA_PATH)
base_env_template = build_base_env(test_df, features, asset_type)

expert_names = [EXPERT_NAME_MAP[i] for i in range(1, 7)]
experts = []
for name in expert_names:
    ckpt_path = CHECKPOINT_DIR  / name.upper() / f'{name.lower()}_expert.pt'
    expert = load_expert_agent(name, ckpt_path, base_env_template, device=DEVICE, expert_cfg=experts_cfg)
    experts.append(expert)

# Load regime model 
regime_model = None
if REGIME_MODEL_PATH.exists():
    print(f"Loading regime model from {REGIME_MODEL_PATH}")
    regime_model = MultiHorizonRegimeClassifier(REGIME_MODEL_PATH)
else:
    print(f"Regime model not found at {REGIME_MODEL_PATH}, proceeding without regime features.")

# Create environment with regime model
allocator_env = AllocatorEnv(
    build_base_env(test_df, features, asset_type),
    experts,
    lookback=lookback,
    regime_model=regime_model,
    regime_feature_cols=list(REGIME_CONFIG['features']),
    regime_window=int(REGIME_CONFIG['model']['window']),
)

# --- Instantiate IQN allocator with hyperparameters from config ---
iqn_params = alloc_cfg['allocator']
allocator = IQNAllocator(
    state_dim=allocator_env.observation_space.shape[0],
    action_dim=6,
    lr=iqn_params.get('lr', 3e-4),
    gamma=iqn_params.get('gamma', 0.995),
    buffer_size=iqn_params.get('buffer_size', 100000),
    batch_size=iqn_params.get('batch_size', 64),
    hidden_dim=iqn_params.get('hidden_dim', 256),
    n_quantiles=iqn_params.get('n_quantiles', 32),
    embedding_dim=iqn_params.get('embedding_dim', 64),
    num_layers=iqn_params.get('num_layers', 2),
    activation=iqn_params.get('activation', 'relu'),
    target_update_freq=0,
    epsilon_start=0,
    epsilon_end=0,
    epsilon_decay=1,
    risk_tau=iqn_params.get('risk_tau', 0.5),
    device=DEVICE,
)
allocator.load(ALLOCATOR_PATH)   # load the saved IQN checkpoint

print(f'Asset: {ASSET} ({asset_type})')
print(f'Features: {len(features)}')
print(f'Allocator observation dim: {allocator_env.observation_space.shape[0]}')
print(f'Allocator checkpoint: {ALLOCATOR_PATH}')

In [ ]:
# =============================================================================
# 1. Run deterministic evaluation episode
# =============================================================================
from tqdm import tqdm

obs, _ = allocator_env.reset()
done = False

equity_curve = [float(base_env_template.initial_balance)]   # starting equity
step_returns = []
positions = []
expert_actions = []          # which expert was selected (0..5)
expert_signals = []          # the actual signal returned by the chosen expert
rewards = []

# Use a progress bar to show evaluation progress
with tqdm(desc="Evaluating", unit="step") as pbar:
    while not done:
        # 1. Allocator picks an expert (greedy, using risk_tau)
        action = allocator.act(obs, deterministic=False)
        
        # 2. Step the environment
        next_obs, reward, terminated, truncated, info = allocator_env.step(action)
        done = terminated or truncated
        
        # 3. Collect data from info
        step_return = float(info.get("step_return", 0.0))
        equity = float(info.get("equity", equity_curve[-1]))
        position_frac = float(abs(info.get("position_frac", 0.0)))
        expert_signal = float(info.get("allocator_expert_signal", 0.0))
        
        # 4. Store
        step_returns.append(step_return)
        equity_curve.append(equity)
        positions.append(position_frac)
        expert_actions.append(action)
        expert_signals.append(expert_signal)
        rewards.append(float(reward))
        
        # 5. Advance
        obs = next_obs
        pbar.update(1)

print(f"Evaluation finished after {len(step_returns)} steps.")

In [ ]:
# =============================================================================
# 2. Compute performance metrics
# =============================================================================
total_return = (equity_curve[-1] - equity_curve[0]) / (abs(equity_curve[0]) + 1e-8)
sharpe = compute_sharpe(step_returns)
sortino = compute_sortino(step_returns)
max_dd = compute_max_drawdown(equity_curve)
avg_position = np.mean(positions) if positions else 0.0
annualized_sharpe = sharpe * np.sqrt(252)   # assuming daily data

# Expert selection statistics
action_counts = np.bincount(expert_actions, minlength=6)
action_freq = action_counts / len(expert_actions)

metrics = {
    "Total return (%)": total_return * 100,
    "Sharpe (daily)": sharpe,
    "Sharpe (annualized)": annualized_sharpe,
    "Sortino": sortino,
    "Max drawdown (%)": max_dd * 100,
    "Average position": avg_position,
    "Number of steps": len(step_returns),
    "Final equity": equity_curve[-1],
}

print("\n" + "=" * 50)
print("EVALUATION METRICS")
print("=" * 50)
for key, val in metrics.items():
    print(f"{key:25s} : {val:>12.4f}")
print("=" * 50)

print("\nExpert selection frequencies:")
for i, freq in enumerate(action_freq):
    print(f"  Expert {i} ({expert_names[i]}): {freq * 100:6.2f}%")

In [ ]:
# =============================================================================
# 3. Visualise equity curve and drawdown
# =============================================================================
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# --- Equity curve ---
axes[0].plot(equity_curve, label="Equity", color="navy", lw=2)
axes[0].axhline(y=equity_curve[0], color="gray", linestyle="--", alpha=0.5)
axes[0].set_ylabel("Equity")
axes[0].set_title(f"Allocator Equity Curve – {ASSET}")
axes[0].legend()
axes[0].grid(True)

# --- Drawdown ---
running_max = np.maximum.accumulate(equity_curve)
drawdown = (running_max - equity_curve) / (running_max + 1e-8)
axes[1].fill_between(range(len(drawdown)), drawdown * 100, color="red", alpha=0.3)
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_title("Drawdown")
axes[1].grid(True)

# --- Expert selection over time ---
axes[2].step(range(len(expert_actions)), expert_actions, where="mid", color="darkgreen", lw=1)
axes[2].set_yticks(range(6))
axes[2].set_yticklabels([f"{i}:{name}" for i, name in enumerate(expert_names)])
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Selected Expert")
axes[2].set_title("Expert Selection Timeline")
axes[2].grid(True, axis="y")

plt.tight_layout()
plt.show()